In [ ]:
"""
freezeup_manual_review.py  (v3 – all obs shown regardless of freeze-up detection)
──────────────────────────────────────────────────────────────────────────────────
Interactive OpenCV tool for manually reviewing and correcting freeze-up dates.
Shows a merged, chronologically sorted time series of both PlanetScope and
Sentinel-2 imagery so that both sources can be used to ground-truth the date.

NEW IN v3
  • Observations from BOTH sensors are shown for every lake-year, even if only
    one sensor (or neither) detected a freeze-up date algorithmically.
  • The output CSV now has separate "planet_freezeup_date" and
    "sentinel_freezeup_date" columns alongside "manual_freezeup_date".
    Either satellite column is left blank when that sensor had no detected date.

PS thumbnails use bands 3-2-1 (BGR natural colour, 3 m).
S2 thumbnails use B04-B03-B02 (RGB natural colour, 10 m); pixels flagged as
cloud / cloud-shadow / cirrus in the SCL band are blanked to 0 before display.

Each cell in the strip is bordered orange (PS) or teal (S2) and labelled with
the sensor tag so the two sources are easy to tell apart at a glance.

CONTROLS
  ← / A       Move freeze-up divider left  (one image)
  → / D       Move freeze-up divider right (one image)
  ]            Toggle "show all" overview (all images at once for scanning)
  Enter        Confirm position and save result, advance to next lake-year
  X            Mark as NO MANUAL DATE (no detectable freeze-up), advance
  B            Go back one lake-year (undo last save)
  O            Toggle lake outline overlay
  Z            Toggle tight zoom (40 m buffer vs 500 m default)
  ESC          Quit (progress already saved up to this point)

Only 8 images are shown at a time so each thumbnail stays large and readable.
Moving the divider past the visible window scrolls the strip by 4 images.
Press ] to expand to the full time series for orientation.

OUTPUT CSV columns
  study_site | lake_id | year |
  planet_freezeup_date | sentinel_freezeup_date |
  manual_freezeup_date | image_before | image_after
"""

import os
import glob
import queue
import threading
import warnings
import csv

import numpy as np
import pandas as pd
import cv2
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
#  USER CONFIG
# ══════════════════════════════════════════════════════════════════════════════

_FREEZEUP_OUT   = r"E:\planetscope_lake_ice\Data\Output\Freezeup"

# ── PlanetScope ──────────────────────────────────────────────────────────────
_PS_FREEZEUP_DATES = r"C:\Users\nj142\Desktop\Time Series\PS\Freezeup"
_PS_INPUT_BASE     = r"E:\planetscope_lake_ice\Data\Input\Study Sites - Manual ALPOD Data"

# ── Sentinel-2 ───────────────────────────────────────────────────────────────
_S2_FREEZEUP_DATES = r"C:\Users\nj142\Desktop\Time Series\S2\Freezeup"
_S2_INPUT_BASE     = r"E:\sentinel_lake_ice\Data\Input\Freezeup S2 Data"

SITES = ["NS", "YF", "YKD"]

PS_FREEZEUP_RESULTS_CSVS = {
    site: os.path.join(_PS_FREEZEUP_DATES, f"{site}_freezeup_dates.csv")
    for site in SITES
}

S2_FREEZEUP_RESULTS_CSVS = {
    site: os.path.join(_S2_FREEZEUP_DATES, f"{site}_freezeup_dates.csv")
    for site in SITES
}

PS_CLOUD_CSVS = [
    os.path.join(_PS_INPUT_BASE, f"{site}_cloud_classifications.csv")
    for site in SITES
]

PS_SITE_FOLDERS = {
    site: os.path.join(_PS_INPUT_BASE, f"{site} 50x50 km")
    for site in SITES
}

# S2 imagery is organised as:  _S2_INPUT_BASE / <SITE> / <acq_folder> / B02.tif …
S2_SITE_FOLDERS = {
    site: os.path.join(_S2_INPUT_BASE, site)
    for site in SITES
}

SITE_SHAPEFILES = {
    site: os.path.join(
        _PS_INPUT_BASE, f"{site} 50x50 km",
        f"{site} Lakes from ALPOD Shapefile", f"{site}_50x50km_lakes.shp"
    ) for site in SITES
}

OUTPUT_CSV = os.path.join(_FREEZEUP_OUT, "manual_freezeup_review.csv")

# ── Display ──────────────────────────────────────────────────────────────────
MAX_WINDOW_W   = 1800
MAX_WINDOW_H   = 900
DIVIDER_W      = 44
LABEL_H        = 62
HEADER_H       = 52
MIN_THUMB_W    = 90

WINDOW_SIZE    = 8
WINDOW_SHIFT   = 4

FREEZE_THRESHOLD   = 75.0
BOUNDARY_BUFFER_M  = 500
ZOOM_TIGHT_BUFFER  = 40
PRELOAD_AHEAD      = 10

# SCL cloud/shadow classes that are blanked in S2 thumbnails
S2_CLOUD_SCL_CLASSES = [3, 8, 9, 10]   # cloud shadow, med cloud, high cloud, cirrus

# ── Colours (BGR) ────────────────────────────────────────────────────────────
C_BG        = (26,  13,  13)
C_CELL      = (38,  22,  22)
C_HEADER    = (42,  20,  10)
C_DIVIDER   = (0,  140, 220)
C_ICE_HIGH  = (70,   70, 255)
C_ICE_LOW   = (220, 170,  60)
C_WHITE     = (255, 255, 255)
C_DIM       = (160, 160, 190)
C_CONFIRMED = (60,  200,  80)
C_WARN      = (50,  180, 255)
C_PS_LABEL  = (0,   140, 255)   # orange  – PlanetScope
C_S2_LABEL  = (180, 200,  30)   # teal    – Sentinel-2
C_NO_DATE   = (110, 110, 140)   # grey    – sensor had no detected date
SENSOR_BORDER_H = 4             # px coloured border at top of each cell


# ══════════════════════════════════════════════════════════════════════════════
#  UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def _norm_id(v):
    s = str(v)
    return s[:-2] if s.endswith(".0") else s


def _strip_tif_suffix(fname):
    return fname.rsplit("_ortho_analytic", 1)[0]


def _stretch(band, lo=2, hi=98):
    band  = band.astype(np.float32)
    valid = band > 0
    if not valid.any():
        return np.zeros_like(band, dtype=np.uint8)
    p_lo, p_hi = np.percentile(band[valid], (lo, hi))
    if p_hi <= p_lo:
        return np.zeros_like(band, dtype=np.uint8)
    out = np.clip((band - p_lo) / (p_hi - p_lo) * 255.0, 0, 255)
    out[~valid] = 0
    return out.astype(np.uint8)


def _put_text(img, text, x, y, scale=0.45, colour=C_WHITE, thickness=1):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                scale, colour, thickness, cv2.LINE_AA)


def letterbox_frame(frame, win_w, win_h):
    fh, fw = frame.shape[:2]
    if fw == 0 or fh == 0 or win_w == 0 or win_h == 0:
        return frame
    scale   = min(win_w / fw, win_h / fh)
    new_w   = int(fw * scale)
    new_h   = int(fh * scale)
    interp  = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(frame, (new_w, new_h), interpolation=interp)
    canvas  = np.zeros((win_h, win_w, 3), dtype=np.uint8)
    x0      = (win_w - new_w) // 2
    y0      = (win_h - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized
    return canvas


# ══════════════════════════════════════════════════════════════════════════════
#  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_very_cloudy(cloud_csvs=PS_CLOUD_CSVS):
    """Returns a set of PS TIF filenames classified as Very Cloudy."""
    names = set()
    for path in cloud_csvs:
        if not os.path.exists(path):
            print(f"  WARNING: cloud CSV not found: {path}")
            continue
        df   = pd.read_csv(path)
        mask = df["Classification"].str.strip() == "Very Cloudy"
        for fname in df.loc[mask, "Filename"].tolist():
            names.add(str(fname).strip())
    print(f"  {len(names):,} Very Cloudy PS filenames loaded.")
    return names


def load_results(results_csvs, sensor_tag):
    """
    Load results CSVs for one sensor (PS or S2).
    Returns a DataFrame with a 'sensor' column added.

    NOTE (v3): ALL rows are returned — including those without a freeze-up date
    — so that their observations can still be displayed as ground-truth imagery.
    """
    frames = []
    for site, path in results_csvs.items():
        if not os.path.exists(path):
            print(f"  WARNING: {sensor_tag} results CSV not found: {path}")
            continue
        df = pd.read_csv(path)
        df["study_site"] = site
        df["sensor"]     = sensor_tag
        df["lake_id"]    = df["lake_id"].astype(str).str.replace(r"\.0$", "", regex=True)
        df["year"]       = pd.to_numeric(df["year"], errors="coerce")
        df["freezeup_date"] = pd.to_datetime(df["freezeup_date"], errors="coerce")
        if "obs_before_date" in df.columns:
            df["obs_before_date"] = pd.to_datetime(df["obs_before_date"], errors="coerce")
        df = df[df["year"].notna()].copy()
        df["year"] = df["year"].astype(int)
        frames.append(df)
        n_with = df["freezeup_date"].notna().sum()
        print(f"    {sensor_tag} {site}: {len(df):,} rows total  "
              f"({n_with:,} with detected freeze-up date, "
              f"{len(df) - n_with:,} without)")
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    # Return ALL rows — freeze-up date may be NaT
    return out.reset_index(drop=True)


# ══════════════════════════════════════════════════════════════════════════════
#  OBS PARSING FROM ObsK COLUMNS
# ══════════════════════════════════════════════════════════════════════════════

def parse_obs_from_row(fu_row, very_cloudy, sensor_tag):
    """
    Walk Obs1…ObsN columns and return (obs_list, cloudy_skipped).

    For PS:  ObsK is a full TIF filename; very_cloudy is checked directly.
    For S2:  ObsK is an acquisition folder name (e.g. S2A_4WFD_20201017_1_L2A);
             very_cloudy is an empty set (cloud masking is per-pixel via SCL).

    Each entry in obs_list is a dict with keys:
        sensor, image_name, datetime, ice_fraction, water_fraction, tif_filename
    """
    obs_list       = []
    cloudy_skipped = 0
    k = 1
    while True:
        fname_col = f"Obs{k}"
        date_col  = f"Obs{k}Date"
        ice_col   = f"Obs{k}PctIce"
        wat_col   = f"Obs{k}PctWat"

        if fname_col not in fu_row:
            break

        fname = fu_row.get(fname_col)
        if pd.isna(fname) or str(fname).strip() == "":
            break
        fname = str(fname).strip()

        if fname in very_cloudy:
            cloudy_skipped += 1
            k += 1
            continue

        date_val = fu_row.get(date_col)
        try:
            dt = pd.Timestamp(date_val)
        except Exception:
            dt = pd.NaT

        def _safe_float(v):
            try:
                return float(v)
            except (TypeError, ValueError):
                return 0.0

        ice = _safe_float(fu_row.get(ice_col, 0.0))
        wat = _safe_float(fu_row.get(wat_col, 0.0))

        obs_list.append({
            "sensor":          sensor_tag,
            "image_name":      fname,
            "datetime":        dt,
            "ice_fraction":    ice,
            "water_fraction":  wat,
            "tif_filename":    fname,   # resolved later
        })
        k += 1

    return obs_list, cloudy_skipped


# ══════════════════════════════════════════════════════════════════════════════
#  IMAGE LOOKUPS
# ══════════════════════════════════════════════════════════════════════════════

def build_ps_image_lookup(site_folder):
    """Recursively index every PS .tif by basename and stripped stem."""
    lookup = {}
    for p in glob.glob(os.path.join(site_folder, "**", "*.tif"), recursive=True):
        fname = os.path.basename(p)
        lookup.setdefault(fname, p)
        lookup.setdefault(_strip_tif_suffix(fname), p)
    return lookup


def build_s2_image_lookup(s2_site_folder):
    """
    Index S2 acquisition folders.
    Returns {folder_name: folder_path}.
    Each folder contains B02.tif, B03.tif, B04.tif, SCL.tif, etc.
    """
    lookup = {}
    if not os.path.isdir(s2_site_folder):
        return lookup
    for entry in os.scandir(s2_site_folder):
        if entry.is_dir():
            lookup[entry.name] = entry.path
    return lookup


def build_lake_year_list(df_ps, df_s2, sample_per_site=250, seed=42):
    """
    Build a combined list of lake-years to review, sampling up to
    ``sample_per_site`` per study-site.

    (v3) Only lake-years where AT LEAST ONE sensor has a detected freeze-up
    date are eligible.  Lake-years where neither PS nor S2 produced a date
    are excluded from sampling entirely — the imagery from the sensor(s) that
    DO have observations is still shown for the included lake-years, giving
    full ground-truth context even when only one sensor fired.
    """
    from collections import defaultdict

    # ── Step 1: collect every lake-year that has at least one detected date ──
    keys_with_date: set = set()
    for df in [df_ps, df_s2]:
        if df.empty:
            continue
        sub = df[df["freezeup_date"].notna()]
        for _, r in sub.iterrows():
            keys_with_date.add(
                (_norm_id(r["lake_id"]), int(r["year"]), str(r["study_site"]))
            )

    if not keys_with_date:
        print("  WARNING: no lake-years have a detected freeze-up date "
              "in either sensor.")
        return []

    print(f"  {len(keys_with_date):,} unique lake-years have at least one "
          f"detected freeze-up date (across both sensors).")

    # ── Step 2: group by site and sample up to sample_per_site ───────────────
    by_site: dict = defaultdict(list)
    for key in keys_with_date:
        by_site[key[2]].append(key)

    rng     = np.random.default_rng(seed)
    sampled = set()
    for site, keys in sorted(by_site.items()):
        n = min(len(keys), sample_per_site)
        chosen = rng.choice(len(keys), size=n, replace=False)
        for i in chosen:
            sampled.add(keys[i])
        print(f"    {site}: sampled {n} of {len(keys):,} eligible lake-years")

    rows = sorted(sampled, key=lambda t: (t[2], t[0], t[1]))
    return [{"lake_id": lid, "year": yr, "study_site": ss}
            for lid, yr, ss in rows]


# ══════════════════════════════════════════════════════════════════════════════
#  THUMBNAIL LOADERS
# ══════════════════════════════════════════════════════════════════════════════

def _compute_rings(lake_proj, out_transform, clip_w, clip_h, thumb_w, thumb_h):
    """Shared ring-computation logic for both PS and S2."""
    inv_t   = ~out_transform
    scale_x = thumb_w / clip_w
    scale_y = thumb_h / clip_h
    polys   = (lake_proj.geoms if lake_proj.geom_type == "MultiPolygon"
               else [lake_proj])
    rings   = []
    for poly in polys:
        for ring in [poly.exterior, *poly.interiors]:
            xs   = np.array([c[0] for c in ring.coords])
            ys   = np.array([c[1] for c in ring.coords])
            cols = inv_t.a * xs + inv_t.b * ys + inv_t.c
            rows = inv_t.d * xs + inv_t.e * ys + inv_t.f
            pts  = np.column_stack(
                (np.round(cols * scale_x).astype(np.int32),
                 np.round(rows * scale_y).astype(np.int32))
            )
            rings.append(pts)
    return rings


def load_ps_thumbnail(tif_path, lake_geom, lake_crs, thumb_w, thumb_h,
                      buffer_m=BOUNDARY_BUFFER_M):
    """Original PS thumbnail loader (bands 3-2-1, BGR natural colour)."""
    try:
        with rasterio.open(tif_path) as src:
            lake_proj = (gpd.GeoSeries([lake_geom], crs=lake_crs)
                         .to_crs(src.crs).geometry.iloc[0])

            try:
                lake_only, _ = rio_mask(src, [lake_proj],
                                        crop=True, nodata=0, filled=True)
            except Exception:
                return None, [], False
            if lake_only.size == 0 or not np.any(lake_only > 0):
                return None, [], False

            out_img, out_transform = rio_mask(src, [lake_proj.buffer(buffer_m)],
                                              crop=True, nodata=0)
            if out_img.shape[0] < 3:
                return None, [], True
            clip_h = out_img.shape[1]
            clip_w = out_img.shape[2]
            rgb = np.dstack([_stretch(out_img[2]),
                             _stretch(out_img[1]),
                             _stretch(out_img[0])])

        bgr   = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        thumb = cv2.resize(bgr, (thumb_w, thumb_h), interpolation=cv2.INTER_AREA)
        rings = _compute_rings(lake_proj, out_transform, clip_w, clip_h,
                               thumb_w, thumb_h)
        return thumb, rings, True

    except Exception as exc:
        print(f"    PS thumbnail error ({os.path.basename(tif_path)}): {exc}")
        return None, [], True


def load_s2_thumbnail(folder_path, lake_geom, lake_crs, thumb_w, thumb_h,
                      buffer_m=BOUNDARY_BUFFER_M):
    """
    Load an S2 thumbnail from a per-acquisition folder containing B02/B03/B04/SCL.tif.

    RGB = B04 (red) / B03 (green) / B02 (blue), 10 m resolution.
    SCL cloud classes 3/8/9/10 are blanked to 0 before display.
    """
    b04_path = os.path.join(folder_path, "B04.tif")
    b03_path = os.path.join(folder_path, "B03.tif")
    b02_path = os.path.join(folder_path, "B02.tif")
    scl_path = os.path.join(folder_path, "SCL.tif")

    for p in [b04_path, b03_path, b02_path]:
        if not os.path.exists(p):
            return None, [], True     # not found → show placeholder, don't drop

    try:
        # ── reproject lake to B04 CRS (all 10 m bands share the same CRS) ──
        with rasterio.open(b04_path) as src:
            lake_proj = (gpd.GeoSeries([lake_geom], crs=lake_crs)
                         .to_crs(src.crs).geometry.iloc[0])
            buffered  = lake_proj.buffer(buffer_m)

            # Validity check (un-buffered lake only)
            try:
                lake_only, _ = rio_mask(src, [lake_proj],
                                        crop=True, nodata=0, filled=True)
            except Exception:
                return None, [], False
            if lake_only.size == 0 or not np.any(lake_only > 0):
                return None, [], False

            r4, out_transform = rio_mask(src, [buffered], crop=True, nodata=0)
            clip_h, clip_w    = r4.shape[1], r4.shape[2]

        with rasterio.open(b03_path) as src:
            r3, _ = rio_mask(src, [buffered], crop=True, nodata=0)

        with rasterio.open(b02_path) as src:
            r2, _ = rio_mask(src, [buffered], crop=True, nodata=0)

        # ── SCL cloud mask (20 m → resample to match 10 m clip size) ──────
        cloud_mask_2d = None
        if os.path.exists(scl_path):
            try:
                with rasterio.open(scl_path) as scl_src:
                    lake_proj_scl = (gpd.GeoSeries([lake_geom], crs=lake_crs)
                                     .to_crs(scl_src.crs).geometry.iloc[0])
                    scl_data, _ = rio_mask(scl_src,
                                           [lake_proj_scl.buffer(buffer_m)],
                                           crop=True, nodata=0)
                    scl_band      = scl_data[0].astype(np.uint8)
                    cloud_mask_2d = cv2.resize(scl_band, (clip_w, clip_h),
                                               interpolation=cv2.INTER_NEAREST)
            except Exception as e:
                print(f"    SCL warning ({os.path.basename(folder_path)}): {e}")

        r4b = r4[0].copy().astype(np.float32)
        r3b = r3[0].copy().astype(np.float32)
        r2b = r2[0].copy().astype(np.float32)

        if cloud_mask_2d is not None:
            cloud = np.isin(cloud_mask_2d, S2_CLOUD_SCL_CLASSES)
            r4b[cloud] = 0
            r3b[cloud] = 0
            r2b[cloud] = 0

        rgb   = np.dstack([_stretch(r4b), _stretch(r3b), _stretch(r2b)])
        bgr   = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        thumb = cv2.resize(bgr, (thumb_w, thumb_h), interpolation=cv2.INTER_AREA)
        rings = _compute_rings(lake_proj, out_transform, clip_w, clip_h,
                               thumb_w, thumb_h)
        return thumb, rings, True

    except Exception as exc:
        print(f"    S2 thumbnail error ({os.path.basename(folder_path)}): {exc}")
        return None, [], True


# ══════════════════════════════════════════════════════════════════════════════
#  LAKE-YEAR CONTAINER
# ══════════════════════════════════════════════════════════════════════════════

class LakeYearData:
    __slots__ = ("lake_id", "year", "study_site",
                 "obs", "thumbs", "outlines",
                 "lake_geom", "lake_crs",
                 "ps_img_lookup", "s2_img_lookup",
                 # Per-sensor detected freeze-up dates (may be pd.NaT)
                 "ps_freezeup_date", "s2_freezeup_date",
                 "divider_pos",
                 "thumb_w", "thumb_h",
                 "window_start", "show_all",
                 "ps_cloudy_skipped")

    def __init__(self, **kw):
        self.window_start      = 0
        self.show_all          = False
        self.ps_cloudy_skipped = 0
        for k, v in kw.items():
            setattr(self, k, v)

    @property
    def n(self):
        return len(self.obs)

    def _max_window_start(self):
        return max(0, self.n - WINDOW_SIZE)

    def _ensure_window_visible(self):
        if self.show_all or self.n <= WINDOW_SIZE:
            self.window_start = 0
            return
        rel = self.divider_pos - self.window_start
        if rel < 0:
            self.window_start = max(0, self.window_start - WINDOW_SHIFT)
        elif rel >= WINDOW_SIZE:
            self.window_start = min(self._max_window_start(),
                                    self.window_start + WINDOW_SHIFT)
        self.window_start = max(0, min(self._max_window_start(), self.window_start))

    def center_window_on_divider(self):
        if self.n <= WINDOW_SIZE:
            self.window_start = 0
            return
        target = max(self.divider_pos, 0) - (WINDOW_SIZE // 2 - 1)
        self.window_start = max(0, min(self._max_window_start(), target))

    def move_left(self):
        self.divider_pos = max(-1, self.divider_pos - 1)
        self._ensure_window_visible()

    def move_right(self):
        self.divider_pos = min(self.n - 1, self.divider_pos + 1)
        self._ensure_window_visible()

    def toggle_show_all(self):
        self.show_all = not self.show_all
        if not self.show_all:
            self.center_window_on_divider()


# ══════════════════════════════════════════════════════════════════════════════
#  PREPARE  (worker-safe)
# ══════════════════════════════════════════════════════════════════════════════

def prepare_lake_year(task, very_cloudy,
                      df_ps_results, df_s2_results,
                      ps_img_lookups, s2_img_lookups,
                      shp_cache):
    lake_id    = task["lake_id"]
    year       = task["year"]
    study_site = task["study_site"]

    def _get_fu_row(df):
        if df.empty:
            return None
        rows = df[(df["lake_id"]    == lake_id) &
                  (df["year"]       == year)    &
                  (df["study_site"] == study_site)]
        return rows.iloc[0].to_dict() if not rows.empty else None

    fu_ps = _get_fu_row(df_ps_results)
    fu_s2 = _get_fu_row(df_s2_results)

    # ── Gather observations from each sensor ─────────────────────────────────
    # (v3) We collect obs from BOTH sensors regardless of whether they have a
    # detected freeze-up date.  A sensor row without a freeze-up date is still
    # parsed for its ObsK columns so all available imagery is displayed.
    all_obs        = []
    ps_cloudy_skip = 0

    if fu_ps is not None:
        ps_obs, ps_cloudy_skip = parse_obs_from_row(fu_ps, very_cloudy, "PS")
        all_obs.extend(ps_obs)

    if fu_s2 is not None:
        # S2 cloud filtering is per-pixel via SCL; pass empty set here
        s2_obs, _ = parse_obs_from_row(fu_s2, set(), "S2")
        all_obs.extend(s2_obs)

    if not all_obs:
        print(f"    SKIP REASON: no valid obs for {lake_id}/{year}/{study_site}  "
              f"(ps_cloudy_skipped={ps_cloudy_skip})")
        return None

    # Sort by datetime across both sensors
    all_obs.sort(key=lambda r: r["datetime"] if pd.notna(r["datetime"])
                 else pd.Timestamp.min)

    # ── Extract per-sensor freeze-up dates (NaT if not detected) ─────────────
    def _extract_date(fu_row, key="freezeup_date"):
        if fu_row is None:
            return pd.NaT
        val = fu_row.get(key)
        if val is None or (not isinstance(val, pd.Timestamp) and pd.isna(val)):
            return pd.NaT
        try:
            return pd.Timestamp(val)
        except Exception:
            return pd.NaT

    ps_freezeup_date = _extract_date(fu_ps, "freezeup_date")
    s2_freezeup_date = _extract_date(fu_s2, "freezeup_date")

    # ── Shapefile ─────────────────────────────────────────────────────────────
    shp_path = SITE_SHAPEFILES.get(study_site)
    if not shp_path or not os.path.exists(shp_path):
        print(f"    SKIP REASON: shapefile not found: {shp_path}")
        return None
    if shp_path not in shp_cache:
        shp_cache[shp_path] = gpd.read_file(shp_path)
    gdf    = shp_cache[shp_path]
    id_col = next((c for c in ("id", "ID", "lake_id", "LAKE_ID", "FID", "OBJECTID")
                   if c in gdf.columns), None)
    if id_col is None:
        print(f"    SKIP REASON: no recognised ID column in shapefile "
              f"(cols: {list(gdf.columns)})")
        return None
    lake_rows = gdf[gdf[id_col].apply(_norm_id) == lake_id]
    if lake_rows.empty:
        print(f"    SKIP REASON: lake_id {lake_id!r} not found in shapefile "
              f"column {id_col!r}")
        return None
    lake_geom = lake_rows.geometry.iloc[0]
    lake_crs  = gdf.crs

    ps_lookup = ps_img_lookups.get(study_site, {})
    s2_lookup = s2_img_lookups.get(study_site, {})

    # Resolve tif_filename for each obs
    for row in all_obs:
        if row["sensor"] == "PS":
            tif_path = ps_lookup.get(row["image_name"])
            row["tif_filename"] = os.path.basename(tif_path) if tif_path else row["image_name"]
        else:
            folder_path = s2_lookup.get(row["image_name"])
            row["tif_filename"] = os.path.basename(folder_path) if folder_path else row["image_name"]

    # ── Thumbnail sizing ───────────────────────────────────────────────────────
    n           = len(all_obs)
    available_w = MAX_WINDOW_W - DIVIDER_W
    visible_n   = min(WINDOW_SIZE, n) if n > 0 else 1
    cell_w      = max(MIN_THUMB_W, available_w // visible_n)
    thumb_w     = cell_w
    thumb_h     = max(80, int(280 * cell_w / 280))
    cell_h      = thumb_h + LABEL_H
    total_h     = HEADER_H + cell_h
    if total_h > MAX_WINDOW_H:
        scale   = (MAX_WINDOW_H - HEADER_H - LABEL_H) / thumb_h
        thumb_h = max(80, int(thumb_h * scale))
        thumb_w = max(MIN_THUMB_W, int(thumb_w * scale))

    # ── Load thumbnails, dropping obs where lake is entirely under nodata ──────
    filtered_obs, thumbs, outlines = [], [], []
    n_dropped = 0
    for row in all_obs:
        if row["sensor"] == "PS":
            tif_path = ps_lookup.get(row["image_name"])
            if tif_path:
                t, r, lake_valid = load_ps_thumbnail(
                    tif_path, lake_geom, lake_crs, thumb_w, thumb_h)
            else:
                t, r, lake_valid = None, [], True
        else:
            folder_path = s2_lookup.get(row["image_name"])
            if folder_path:
                t, r, lake_valid = load_s2_thumbnail(
                    folder_path, lake_geom, lake_crs, thumb_w, thumb_h)
            else:
                t, r, lake_valid = None, [], True

        if not lake_valid:
            n_dropped += 1
            continue
        filtered_obs.append(row)
        thumbs.append(t)
        outlines.append(r)

    if n_dropped:
        print(f"    Dropped {n_dropped} obs with lake under nodata border.")

    if not filtered_obs:
        print(f"    SKIP REASON: all {len(all_obs)} obs dropped (lake under nodata) "
              f"for {lake_id}/{year}/{study_site}")
        return None

    # ── Divider: place based on PS date first, fall back to S2 ───────────────
    # (v3) A sensor may have obs but no freeze-up date; use whichever date is
    # available to give a sensible starting position for the divider.
    ref_date = pd.NaT
    if pd.notna(ps_freezeup_date):
        # Prefer obs_before_date if PS has it, else use freezeup_date itself
        obs_before = _extract_date(fu_ps, "obs_before_date")
        ref_date   = obs_before if pd.notna(obs_before) else ps_freezeup_date
    elif pd.notna(s2_freezeup_date):
        obs_before = _extract_date(fu_s2, "obs_before_date")
        ref_date   = obs_before if pd.notna(obs_before) else s2_freezeup_date

    divider_pos = -1
    if pd.notna(ref_date):
        ref = pd.Timestamp(ref_date).normalize()
        for i, row in enumerate(filtered_obs):
            if pd.notna(row["datetime"]) and \
               pd.Timestamp(row["datetime"]).normalize() <= ref:
                divider_pos = i

    divider_pos = max(-1, min(divider_pos, len(filtered_obs) - 1))

    lyd = LakeYearData(
        lake_id=lake_id, year=year, study_site=study_site,
        obs=filtered_obs, thumbs=thumbs, outlines=outlines,
        lake_geom=lake_geom, lake_crs=lake_crs,
        ps_img_lookup=ps_lookup, s2_img_lookup=s2_lookup,
        ps_freezeup_date=ps_freezeup_date,
        s2_freezeup_date=s2_freezeup_date,
        divider_pos=divider_pos,
        thumb_w=thumb_w, thumb_h=thumb_h,
        ps_cloudy_skipped=ps_cloudy_skip,
    )
    lyd.center_window_on_divider()
    return lyd


def preload_worker(task_q, result_q, very_cloudy,
                   df_ps_results, df_s2_results,
                   ps_img_lookups, s2_img_lookups, shp_cache):
    while True:
        task = task_q.get()
        if task is None:
            break
        key  = (_norm_id(task["lake_id"]), task["year"], task["study_site"])
        data = None
        try:
            data = prepare_lake_year(task, very_cloudy,
                                     df_ps_results, df_s2_results,
                                     ps_img_lookups, s2_img_lookups,
                                     shp_cache)
        except Exception as exc:
            print(f"  Pre-load error {key}: {exc}")
        result_q.put((key, data))
        task_q.task_done()


# ══════════════════════════════════════════════════════════════════════════════
#  ZOOM RELOAD
# ══════════════════════════════════════════════════════════════════════════════

def _reload_thumbs(lyd, buffer_m):
    new_thumbs, new_outlines = [], []
    for row in lyd.obs:
        if row["sensor"] == "PS":
            tif_path = lyd.ps_img_lookup.get(row["image_name"])
            if tif_path:
                t, r, _ = load_ps_thumbnail(tif_path, lyd.lake_geom, lyd.lake_crs,
                                            lyd.thumb_w, lyd.thumb_h, buffer_m)
            else:
                t, r = None, []
        else:
            folder_path = lyd.s2_img_lookup.get(row["image_name"])
            if folder_path:
                t, r, _ = load_s2_thumbnail(folder_path, lyd.lake_geom, lyd.lake_crs,
                                            lyd.thumb_w, lyd.thumb_h, buffer_m)
            else:
                t, r = None, []
        new_thumbs.append(t)
        new_outlines.append(r)
    lyd.thumbs   = new_thumbs
    lyd.outlines = new_outlines


# ══════════════════════════════════════════════════════════════════════════════
#  RENDERING
# ══════════════════════════════════════════════════════════════════════════════

def _mid_date(lyd):
    dp = lyd.divider_pos
    n  = lyd.n
    b  = lyd.obs[dp]         if 0 <= dp < n     else None
    a  = lyd.obs[dp + 1]     if 0 <= dp + 1 < n else None
    if b is None and a is None:
        return None
    if b is None:
        return pd.Timestamp(a["datetime"])
    if a is None:
        return pd.Timestamp(b["datetime"])
    t0, t1 = pd.Timestamp(b["datetime"]), pd.Timestamp(a["datetime"])
    return t0 + (t1 - t0) / 2


def _flanking_fnames(lyd):
    dp = lyd.divider_pos
    n  = lyd.n
    bf = str(lyd.obs[dp]["tif_filename"])     if 0 <= dp < n     else ""
    af = str(lyd.obs[dp + 1]["tif_filename"]) if 0 <= dp + 1 < n else ""
    return bf, af


def _draw_divider(frame, x, y, w, h, lyd, man_s):
    cv2.rectangle(frame, (x, y), (x + w - 1, y + h - 1), C_DIVIDER, -1)
    tx = x + w // 2 - 7
    for j, ch in enumerate("FREEZE"):
        _put_text(frame, ch, tx, y + 22 + j * 18, scale=0.40,
                  colour=C_WHITE, thickness=1)
    _put_text(frame, "UP", tx - 2, y + 22 + 6 * 18, scale=0.40, colour=C_WHITE)
    _put_text(frame, man_s[:6], tx - 8, y + h - 28, scale=0.35,
              colour=(220, 220, 255))
    _put_text(frame, man_s[7:] if len(man_s) > 6 else "", tx - 5, y + h - 14,
              scale=0.35, colour=(220, 220, 255))


def render_frame(lyd, current_idx, total, show_outline=True, zoom_tight=False):
    n  = lyd.n
    dp = lyd.divider_pos

    # ── Visible slice & per-thumb display size ───────────────────────────────
    if lyd.show_all or n <= WINDOW_SIZE:
        vis_start = 0
        vis_count = n
        if n > 0:
            avail  = MAX_WINDOW_W - DIVIDER_W
            target = max(4, avail // max(n, 1))
            disp_w = min(lyd.thumb_w, target)
        else:
            disp_w = lyd.thumb_w
        disp_h = (int(lyd.thumb_h * disp_w / lyd.thumb_w)
                  if lyd.thumb_w > 0 else lyd.thumb_h)
    else:
        vis_start = lyd.window_start
        vis_count = min(WINDOW_SIZE, n - vis_start)
        disp_w    = lyd.thumb_w
        disp_h    = lyd.thumb_h

    show_labels   = disp_w >= 60
    show_outlines = show_outline and disp_w >= 50

    cell_h   = (disp_h + LABEL_H) if show_labels else disp_h
    canvas_w = max(MAX_WINDOW_W // 4, vis_count * disp_w + DIVIDER_W)
    canvas_h = HEADER_H + cell_h + 4

    frame = np.full((canvas_h, canvas_w, 3), C_BG, dtype=np.uint8)

    # ── Header ───────────────────────────────────────────────────────────────
    frame[:HEADER_H, :] = C_HEADER

    # Per-sensor detected dates (greyed out as "none" if not detected)
    def _fmt_date(ts):
        return pd.Timestamp(ts).strftime("%b %d %Y") if pd.notna(ts) else "none"

    ps_date_s = _fmt_date(lyd.ps_freezeup_date)
    s2_date_s = _fmt_date(lyd.s2_freezeup_date)

    mid   = _mid_date(lyd)
    man_s = mid.strftime("%b %d %Y") if mid is not None else (
        "before 1st obs" if dp == -1 else "?")

    if lyd.show_all or n <= WINDOW_SIZE:
        view_s = f"showing all {n}"
    else:
        last_visible = min(vis_start + WINDOW_SIZE, n)
        view_s = f"images {vis_start + 1}-{last_visible} of {n}"

    n_ps = sum(1 for o in lyd.obs if o["sensor"] == "PS")
    n_s2 = sum(1 for o in lyd.obs if o["sensor"] == "S2")
    sensor_s = f"PS:{n_ps}  S2:{n_s2}"

    cloudy_s = (f"   !! {lyd.ps_cloudy_skipped} PS cloudy skipped !!"
                if lyd.ps_cloudy_skipped > 0 else "")

    line1 = (f"  Lake {lyd.lake_id}   {lyd.year}   {lyd.study_site}     "
             f"PS detected: {ps_date_s}     "
             f"S2 detected: {s2_date_s}     "
             f"Manual: {man_s}     "
             f"[{view_s}]  ({sensor_s})  ({current_idx} / {total})"
             f"{cloudy_s}")

    zoom_s    = "ON (40m)" if zoom_tight   else "off"
    outline_s = "ON"       if show_outline else "off"
    showall_s = "ON"       if lyd.show_all else "off"
    line2 = (f"  Controls:  <- / A  left    -> / D  right    "
             f"] = show all [{showall_s}]    "
             f"Enter = save    X = no date    B = back    "
             f"O = outline [{outline_s}]    Z = zoom [{zoom_s}]    ESC = quit    "
             f"[orange border=PS  teal border=S2]")

    # Highlight the header row if either sensor has no detected date
    neither_detected = pd.isna(lyd.ps_freezeup_date) and pd.isna(lyd.s2_freezeup_date)
    one_missing      = (pd.isna(lyd.ps_freezeup_date) != pd.isna(lyd.s2_freezeup_date))
    header_col = (C_WARN if lyd.ps_cloudy_skipped > 0
                  else C_WHITE if not (neither_detected or one_missing)
                  else (180, 140, 255) if neither_detected   # purple: both missing
                  else (200, 200, 100))                       # yellow: one missing
    _put_text(frame, line1, 6, 18, scale=0.50, colour=header_col)
    _put_text(frame, line2, 6, 40, scale=0.40, colour=C_DIM)

    # Colour-code the per-sensor date labels in the header
    # (PS date label in orange, S2 date label in teal, greyed when absent)
    # The labels are already embedded in line1 as plain text; the colour coding
    # is conveyed by the cell border colours below.

    # ── Body: thumbnails + divider ───────────────────────────────────────────
    x  = 0
    cy = HEADER_H

    need_resize   = (disp_w != lyd.thumb_w) or (disp_h != lyd.thumb_h)
    outline_scale = (disp_w / lyd.thumb_w, disp_h / lyd.thumb_h) \
                    if lyd.thumb_w > 0 else (1.0, 1.0)

    for i_local in range(vis_count):
        i_global = vis_start + i_local

        if i_global == dp + 1:
            _draw_divider(frame, x, cy, DIVIDER_W, cell_h, lyd, man_s)
            x += DIVIDER_W

        frame[cy:cy + cell_h, x:x + disp_w] = C_CELL

        obs_row = lyd.obs[i_global]
        sensor  = obs_row["sensor"]
        border_col = C_S2_LABEL if sensor == "S2" else C_PS_LABEL

        # Coloured sensor border at top of each cell
        frame[cy:cy + SENSOR_BORDER_H, x:x + disp_w] = border_col

        thumb = lyd.thumbs[i_global]
        if thumb is not None:
            t = cv2.resize(thumb, (disp_w, disp_h),
                           interpolation=cv2.INTER_AREA) if need_resize else thumb

            if show_outlines and lyd.outlines[i_global]:
                t = t.copy()
                for pts in lyd.outlines[i_global]:
                    spts = (pts * np.array(outline_scale)).astype(np.int32) \
                           if need_resize else pts
                    cv2.polylines(t, [spts], isClosed=True,
                                  color=(0, 0, 0), thickness=3, lineType=cv2.LINE_AA)
                for pts in lyd.outlines[i_global]:
                    spts = (pts * np.array(outline_scale)).astype(np.int32) \
                           if need_resize else pts
                    cv2.polylines(t, [spts], isClosed=True,
                                  color=(0, 255, 255), thickness=1, lineType=cv2.LINE_AA)

            frame[cy + SENSOR_BORDER_H:cy + disp_h, x:x + disp_w] = \
                t[SENSOR_BORDER_H:, :]
        elif show_labels:
            _put_text(frame, "NOT FOUND",
                      x + disp_w // 2 - 35, cy + disp_h // 2,
                      scale=0.45, colour=(100, 100, 160))

        if show_labels:
            ly = cy + disp_h
            dt_str = (pd.Timestamp(obs_row["datetime"]).strftime("%b %d %Y")
                      if pd.notna(obs_row["datetime"]) else "???")
            _put_text(frame, dt_str, x + 3, ly + 16, scale=0.44, colour=C_WHITE)

            # Sensor tag (right-aligned in label row)
            sensor_label_col = C_S2_LABEL if sensor == "S2" else C_PS_LABEL
            _put_text(frame, sensor, x + disp_w - 28, ly + 16,
                      scale=0.44, colour=sensor_label_col)

            ice = obs_row.get("ice_fraction")
            if ice is not None and not pd.isna(ice):
                col = C_ICE_HIGH if ice >= FREEZE_THRESHOLD else C_ICE_LOW
                _put_text(frame, f"{ice:.1f}% ice",
                          x + 3, ly + 36, scale=0.44, colour=col)

        x += disp_w

    # Divider after the last visible image when dp is at that boundary
    if vis_count > 0 and dp == vis_start + vis_count - 1:
        _draw_divider(frame, x, cy, DIVIDER_W, cell_h, lyd, man_s)

    return frame


# ══════════════════════════════════════════════════════════════════════════════
#  OUTPUT CSV
# ══════════════════════════════════════════════════════════════════════════════

# (v3) Separate planet / sentinel date columns; manual date is the ground truth.
OUTPUT_COLS = ["study_site", "lake_id", "year",
               "planet_freezeup_date", "sentinel_freezeup_date",
               "manual_freezeup_date",
               "image_before", "image_after"]


def load_done(output_csv):
    if not os.path.exists(output_csv):
        return set(), []
    df   = pd.read_csv(output_csv)
    done = {(_norm_id(str(r["lake_id"])), int(r["year"]), str(r["study_site"]))
            for _, r in df.iterrows()}
    return done, df.to_dict("records")


def save_row(output_csv, record, existing_rows):
    existing_rows.append(record)
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_COLS)
        writer.writeheader()
        writer.writerows(existing_rows)


# ══════════════════════════════════════════════════════════════════════════════
#  KEY CODES
# ══════════════════════════════════════════════════════════════════════════════

_KEY_LEFT    = {2424832, 65361, 63234, ord('a'), ord('A')}
_KEY_RIGHT   = {2555904, 65363, 63235, ord('d'), ord('D')}
_KEY_ENTER   = {13, 10}
_KEY_ESC     = {27}
_KEY_X       = {ord('x'), ord('X')}
_KEY_B       = {ord('b'), ord('B')}
_KEY_O       = {ord('o'), ord('O')}
_KEY_Z       = {ord('z'), ord('Z')}
_KEY_BRACKET = {ord(']')}


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def run():
    print("=" * 60)

    print("\nChecking configured paths ...")
    ok = True
    for ds, p in SITE_SHAPEFILES.items():
        exists = os.path.exists(p)
        print(f"  [{'OK' if exists else 'MISSING'}]  {ds} shapefile: {p}")
        ok &= exists
    for ds, p in PS_SITE_FOLDERS.items():
        exists = os.path.isdir(p)
        print(f"  [{'OK' if exists else 'MISSING'}]  {ds} PS imagery: {p}")
        ok &= exists
    for ds, p in S2_SITE_FOLDERS.items():
        exists = os.path.isdir(p)
        print(f"  [{'OK' if exists else 'MISSING'}]  {ds} S2 imagery: {p}")
        ok &= exists
    for ds, p in PS_FREEZEUP_RESULTS_CSVS.items():
        exists = os.path.exists(p)
        print(f"  [{'OK' if exists else 'MISSING'}]  {ds} PS results CSV: {p}")
    for ds, p in S2_FREEZEUP_RESULTS_CSVS.items():
        exists = os.path.exists(p)
        print(f"  [{'OK' if exists else 'MISSING'}]  {ds} S2 results CSV: {p}")
    if not ok:
        print("\n  One or more required paths are missing.")
        print("  Update the paths in USER CONFIG.")
        return

    print("\nLoading PS cloud filter ...")
    very_cloudy = load_very_cloudy()

    print("\nLoading freeze-up results ...")
    print("  PlanetScope:")
    df_ps = load_results(PS_FREEZEUP_RESULTS_CSVS, "PS")
    print("  Sentinel-2:")
    df_s2 = load_results(S2_FREEZEUP_RESULTS_CSVS, "S2")

    if df_ps.empty and df_s2.empty:
        print("ERROR: No results loaded from either sensor.")
        return

    print(f"\n  PS total rows: {len(df_ps):,}  "
          f"(with detected freeze-up: {df_ps['freezeup_date'].notna().sum():,})")
    print(f"  S2 total rows: {len(df_s2):,}  "
          f"(with detected freeze-up: {df_s2['freezeup_date'].notna().sum():,})")

    print("\nBuilding lake-year list ...")
    all_lys = build_lake_year_list(df_ps, df_s2)
    if not all_lys:
        print("ERROR: No lake-years found in results CSVs.")
        return

    done_set, existing_rows = load_done(OUTPUT_CSV)
    todo = [t for t in all_lys
            if (_norm_id(t["lake_id"]), t["year"], t["study_site"]) not in done_set]

    print(f"  Total: {len(all_lys)}   Already done: {len(done_set)}   "
          f"To review: {len(todo)}")
    if not todo:
        print("All lake-years already reviewed!")
        return

    print("\nIndexing TIF files ...")
    ps_img_lookups = {}
    s2_img_lookups = {}
    for ds in {t["study_site"] for t in todo}:
        ps_folder = PS_SITE_FOLDERS.get(ds)
        if ps_folder and os.path.isdir(ps_folder):
            ps_img_lookups[ds] = build_ps_image_lookup(ps_folder)
            print(f"  PS {ds}: ~{len(ps_img_lookups[ds])//2:,} TIFs indexed")
        else:
            print(f"  WARNING: PS imagery folder not found for {ds}: {ps_folder}")

        s2_folder = S2_SITE_FOLDERS.get(ds)
        if s2_folder and os.path.isdir(s2_folder):
            s2_img_lookups[ds] = build_s2_image_lookup(s2_folder)
            print(f"  S2 {ds}: {len(s2_img_lookups[ds]):,} acquisition folders indexed")
        else:
            print(f"  WARNING: S2 imagery folder not found for {ds}: {s2_folder}")

    N_WORKERS = 4
    shp_cache = {}
    task_q    = queue.Queue()
    result_q  = queue.Queue()

    for _w in range(N_WORKERS):
        t = threading.Thread(
            target=preload_worker,
            args=(task_q, result_q, very_cloudy,
                  df_ps, df_s2,
                  ps_img_lookups, s2_img_lookups, shp_cache),
            daemon=True,
            name=f"PreloadWorker-{_w}",
        )
        t.start()

    n_enqueued = 0
    for i in range(min(PRELOAD_AHEAD, len(todo))):
        task_q.put(todo[i])
        n_enqueued += 1

    preload_cache = {}

    def _drain_results():
        while True:
            try:
                key, data = result_q.get_nowait()
                preload_cache[key] = data
            except queue.Empty:
                break

    def _get_lyd(idx):
        t   = todo[idx]
        key = (_norm_id(t["lake_id"]), t["year"], t["study_site"])
        while key not in preload_cache:
            _drain_results()
            if key not in preload_cache:
                try:
                    k2, data = result_q.get(timeout=0.15)
                    preload_cache[k2] = data
                except queue.Empty:
                    pass
        return preload_cache[key]

    WIN = "Freeze-up Manual Review  [PS=orange  S2=teal]"
    cv2.namedWindow(WIN, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(WIN, min(MAX_WINDOW_W, 1600), MAX_WINDOW_H)

    total = len(todo)
    idx   = 0
    shown = 0

    print("\n" + "=" * 60)
    print("OpenCV window opened.  Orange border = PS,  Teal border = S2.")
    print("Header shows PS and S2 detected dates separately (blank = not detected).")
    print("<- -> move divider,  ] = show all,  Enter confirms,  X = no date,  ESC quits.")
    print("=" * 60)

    def _ts_to_str(ts):
        """Format a Timestamp to YYYY-MM-DD, or return '' if NaT."""
        return pd.Timestamp(ts).strftime("%Y-%m-%d") if pd.notna(ts) else ""

    while idx < total:
        t = todo[idx]
        next_to_enqueue = idx + PRELOAD_AHEAD
        while n_enqueued <= next_to_enqueue and n_enqueued < total:
            task_q.put(todo[n_enqueued])
            n_enqueued += 1

        print(f"  Loading [{idx+1}/{total}]: "
              f"Lake {t['lake_id']}  {t['year']}  {t['study_site']} ...",
              end="  ", flush=True)
        lyd = _get_lyd(idx)
        if lyd is None:
            print("SKIPPED (no data)")
            idx += 1
            continue

        n_ps = sum(1 for o in lyd.obs if o["sensor"] == "PS")
        n_s2 = sum(1 for o in lyd.obs if o["sensor"] == "S2")
        ps_note = (_ts_to_str(lyd.ps_freezeup_date) or "no PS date")
        s2_note = (_ts_to_str(lyd.s2_freezeup_date) or "no S2 date")
        cloudy_note = (f"  [{lyd.ps_cloudy_skipped} PS cloudy obs skipped]"
                       if lyd.ps_cloudy_skipped > 0 else "")
        print(f"{lyd.n} obs ready  (PS={n_ps} [{ps_note}]  "
              f"S2={n_s2} [{s2_note}]).{cloudy_note}")

        show_outline = True
        zoom_tight   = False
        win_w, win_h = min(MAX_WINDOW_W, 1600), MAX_WINDOW_H

        while True:
            frame = render_frame(lyd, idx + 1, total,
                                 show_outline=show_outline, zoom_tight=zoom_tight)

            rect = cv2.getWindowImageRect(WIN)
            if rect[2] > 400 and rect[3] > 400:
                win_w, win_h = rect[2], rect[3]

            display = letterbox_frame(frame, win_w, win_h)
            cv2.imshow(WIN, display)
            key = cv2.waitKeyEx(30)

            if key == -1:
                continue

            if key in _KEY_LEFT:
                lyd.move_left()

            elif key in _KEY_RIGHT:
                lyd.move_right()

            elif key in _KEY_BRACKET:
                lyd.toggle_show_all()

            elif key in _KEY_ENTER:
                mid    = _mid_date(lyd)
                man_s  = mid.strftime("%Y-%m-%d") if mid is not None else ""
                bf, af = _flanking_fnames(lyd)
                record = {
                    "study_site":             lyd.study_site,
                    "lake_id":                lyd.lake_id,
                    "year":                   lyd.year,
                    "planet_freezeup_date":   _ts_to_str(lyd.ps_freezeup_date),
                    "sentinel_freezeup_date": _ts_to_str(lyd.s2_freezeup_date),
                    "manual_freezeup_date":   man_s,
                    "image_before":           bf,
                    "image_after":            af,
                }
                save_row(OUTPUT_CSV, record, existing_rows)
                print(f"  + Saved  {lyd.lake_id}/{lyd.year}  ->  manual:{man_s}  "
                      f"PS:{record['planet_freezeup_date'] or 'none'}  "
                      f"S2:{record['sentinel_freezeup_date'] or 'none'}")
                shown += 1
                idx   += 1
                break

            elif key in _KEY_X:
                record = {
                    "study_site":             lyd.study_site,
                    "lake_id":                lyd.lake_id,
                    "year":                   lyd.year,
                    "planet_freezeup_date":   _ts_to_str(lyd.ps_freezeup_date),
                    "sentinel_freezeup_date": _ts_to_str(lyd.s2_freezeup_date),
                    "manual_freezeup_date":   "NO MANUAL DATE",
                    "image_before":           "",
                    "image_after":            "",
                }
                save_row(OUTPUT_CSV, record, existing_rows)
                print(f"  x Saved  {lyd.lake_id}/{lyd.year}  ->  NO MANUAL DATE")
                shown += 1
                idx   += 1
                break

            elif key in _KEY_B:
                if shown == 0:
                    print("  Nothing to undo this session.")
                else:
                    existing_rows.pop()
                    if existing_rows:
                        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
                            writer = csv.DictWriter(f, fieldnames=OUTPUT_COLS)
                            writer.writeheader()
                            writer.writerows(existing_rows)
                    else:
                        try:
                            os.remove(OUTPUT_CSV)
                        except OSError:
                            pass
                    shown -= 1
                    idx   -= 1
                    prev_t = todo[idx]
                    print(f"  Undone -- back to Lake {prev_t['lake_id']}  "
                          f"{prev_t['year']}  {prev_t['study_site']}")
                    break

            elif key in _KEY_O:
                show_outline = not show_outline

            elif key in _KEY_Z:
                zoom_tight = not zoom_tight
                buf = ZOOM_TIGHT_BUFFER if zoom_tight else BOUNDARY_BUFFER_M
                print(f"  Zoom {'tight (40 m)' if zoom_tight else 'default (500 m)'} "
                      f"-- reloading thumbnails ...", end="  ", flush=True)
                _reload_thumbs(lyd, buf)
                print("done.")

            elif key in _KEY_ESC:
                print(f"\n  ESC -- quitting after {shown} reviewed.")
                cv2.destroyAllWindows()
                for _ in range(N_WORKERS):
                    task_q.put(None)
                return

    cv2.destroyAllWindows()
    for _ in range(N_WORKERS):
        task_q.put(None)
    print(f"\n{'='*60}")
    print(f"  Completed!  {shown} lake-years reviewed.")
    print(f"  Results saved to: {OUTPUT_CSV}")
    print("=" * 60)


if __name__ == "__main__":
    run()


Checking configured paths ...
  [OK]  NS shapefile: E:\planetscope_lake_ice\Data\Input\Study Sites - Manual ALPOD Data\NS 50x50 km\NS Lakes from ALPOD Shapefile\NS_50x50km_lakes.shp
  [OK]  YF shapefile: E:\planetscope_lake_ice\Data\Input\Study Sites - Manual ALPOD Data\YF 50x50 km\YF Lakes from ALPOD Shapefile\YF_50x50km_lakes.shp
  [OK]  YKD shapefile: E:\planetscope_lake_ice\Data\Input\Study Sites - Manual ALPOD Data\YKD 50x50 km\YKD Lakes from ALPOD Shapefile\YKD_50x50km_lakes.shp
  [OK]  NS PS imagery: E:\planetscope_lake_ice\Data\Input\Study Sites - Manual ALPOD Data\NS 50x50 km
  [OK]  YF PS imagery: E:\planetscope_lake_ice\Data\Input\Study Sites - Manual ALPOD Data\YF 50x50 km
  [OK]  YKD PS imagery: E:\planetscope_lake_ice\Data\Input\Study Sites - Manual ALPOD Data\YKD 50x50 km
  [OK]  NS S2 imagery: E:\sentinel_lake_ice\Data\Input\Freezeup S2 Data\NS
  [OK]  YF S2 imagery: E:\sentinel_lake_ice\Data\Input\Freezeup S2 Data\YF
  [OK]  YKD S2 imagery: E:\sentinel_lake_ice\Data\I

error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window_w32.cpp:536: error: (-27:Null pointer) NULL window: 'Freeze-up Manual Review  [PS=orange  S2=teal]' in function 'cvGetWindowRect_W32'
